[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 Medium: Implement LayerNorm

Implement **Layer Normalization** from scratch.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma^2$ are computed over the **last dimension**.

### Signature
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (same size as last dim)
    beta: torch.Tensor,    # shift (same size as last dim)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `F.layer_norm` or `torch.nn.LayerNorm`
- Normalize over the last dimension only
- Must support autograd

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch

In [37]:

def my_layer_norm(x, gamma, beta, eps=1e-5): #!gamma and beta are of size (n_features, ) broadcaste over all samples (and prev. dims)
    mu =  x.mean(dim = -1, keepdim=True) #equal to mean = torch.mean(x, dim = -1).unsqueeze(1) --> x: (N, in_features) while mean: (N,) --> to apply broadcasting we need (N, 1)
    var = x.var(dim = -1, keepdim=True, correction=0)
    x_norm = (x - mu) / torch.sqrt(var + eps)
    # scale = torch.randn(x.shape[-1], requires_grad=True)
    # shift = torch.randn(x.shape[-1], requires_grad=True)

    return gamma * x_norm + beta

In [38]:
# 🧪 Debug
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Your output mean:", out.mean(dim=-1))   # should be ~0
print("Your output std: ", out.std(dim=-1))     # should be ~1
print("Match ref?      ", torch.allclose(out, ref, atol=1e-4))

Your output mean: tensor([-1.4901e-08,  7.4506e-09])
Your output std:  tensor([1.0690, 1.0690])
Match ref?       True


In [39]:
# ✅ SUBMIT
from torch_judge import check
check("layernorm")


🧪 Testing: Implement LayerNorm (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Shape and basic behavior (1.2ms)
  ✅ [2/3] With learned parameters (3.2ms)
  ✅ [3/3] Gradient flow (2.3ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (6.7ms total)
  Progress saved. Run status() to see your dashboard.

